# 01 — Primeira previsão de chamadas

Neste notebook você vai: carregar o volume de chamadas de uma central, olhar os dados, pedir uma previsão ao TimesFM 3.0 e desenhar o resultado.

**Como usar:** clique numa célula e aperte `Shift + Enter` para executá-la, uma de cada vez, de cima para baixo. A primeira previsão baixa o modelo e demora mais.

## 1. Carregar os dados

Usamos o arquivo sintético que acompanha o projeto: chamadas recebidas a cada 30 minutos de uma central fictícia (seg–sex 8h–20h, sáb 8h–14h, domingo fechado). Cada linha é um intervalo; `calls_offered` é o número de chamadas.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from callcenter_lab.loading import detect_columns, prepare_series, resample_series

# Para usar um dataset público, rode antes: python scripts/baixar_datasets.py technion-bank
# e troque o caminho abaixo por ../data/raw/technion-bank/technion-bank.csv
csv_path = Path('..') / 'data' / 'samples' / 'chamadas_sinteticas.csv'
df = pd.read_csv(csv_path)
df.head()

## 2. Preparar a série

`detect_columns` adivinha qual coluna é a data e qual é o volume. `prepare_series` ordena e limpa; `resample_series` garante um ponto a cada 30 minutos. Com `fill='zero'`, um intervalo sem registro vira zero: é o que acontece com a central fechada.

In [ ]:
date_col, value_col = detect_columns(df)
print('Data:', date_col, '| Chamadas:', value_col)

series = resample_series(prepare_series(df, date_col, value_col), '30min', how='sum', fill='zero')
series.iloc[-48 * 14:].plot(figsize=(12, 3), title='Chamadas por 30 minutos (últimas 2 semanas)')
plt.show()

Repare nos dois picos do dia (fim da manhã e meio da tarde), no sábado mais curto e no domingo zerado. É esse padrão semanal que o modelo vai tentar reconhecer.

## 3. Carregar o modelo

`load_forecaster` escolhe sozinho o motor: MLX no Mac com Apple silicon, PyTorch nos demais (na GPU NVIDIA, se houver).

In [ ]:
from callcenter_lab.model import choose_backend, load_forecaster

print('Motor:', choose_backend())
model = load_forecaster()

## 4. Prever o próximo dia (48 intervalos de 30 minutos)

In [ ]:
from callcenter_lab.model import forecast

horizon = 48
result = forecast(model, series.to_numpy(), horizon)
print('Previsão pontual:', result.point.shape, '| quantis:', result.quantiles.shape)

A previsão pontual tem 48 valores (um por intervalo). Os quantis têm 9 colunas: 10%, 20%, … 90%.

## 5. Desenhar

In [ ]:
future = pd.date_range(series.index[-1], periods=horizon + 1, freq='30min')[1:]

fig, ax = plt.subplots(figsize=(12, 4))
series.iloc[-48 * 7:].plot(ax=ax, label='Histórico', color='gray')
ax.plot(future, result.point, label='Previsão TimesFM', color='tab:blue', lw=2)
ax.fill_between(future, result.lower, result.upper, alpha=0.2, label='Faixa 10%-90%')
ax.set_ylabel('Chamadas')
ax.legend()
plt.show()

## Para experimentar

- Troque `horizon` por 96 e por 336 (uma semana). O que acontece com a faixa azul?
- Use só `series.iloc[-96:]` como histórico. A previsão piora?
- Baixe um dataset público (`python scripts/baixar_datasets.py technion-bank`) e troque o `csv_path`.